# Car Color Detection & Traffic Counter
## A Complete System for Traffic Analysis

### What This Project Does:
1. Finds cars in pictures or videos
2. Identifies blue cars (shows red boxes) and other colors (shows blue boxes)
3. Counts total number of cars at a traffic signal
4. Counts people waiting at the signal

### Requirements:
- Python 3.7 or higher
- Webcam (optional, for live video)
- Sample traffic images (included or your own)

In [14]:
# Install required packages for Streamlit app
!pip install streamlit opencv-python numpy pillow matplotlib ultralytics scikit-learn streamlit-option-menu

print("All packages installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 82.0 MB/s eta 0:00:00
All packages installed successfully!


In [24]:
import textwrap

# Create the Streamlit app with correct rectangle colors
app_code = textwrap.dedent("""
import streamlit as st
import cv2
import numpy as np
from ultralytics import YOLO
from sklearn.cluster import KMeans
from PIL import Image
import tempfile
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Page configuration
st.set_page_config(
    page_title="Car Color Detection System",
    page_icon="🚗",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS for better UI
st.markdown('''
    <style>
    .main-header {
        font-size: 2.5rem;
        font-weight: bold;
        color: #1E88E5;
        text-align: center;
        padding: 1rem 0;
    }
    .sub-header {
        font-size: 1.2rem;
        color: #666;
        text-align: center;
        padding-bottom: 1rem;
    }
    .stats-box {
        background-color: #f0f2f6;
        padding: 1rem;
        border-radius: 10px;
        margin: 0.5rem 0;
        text-align: center;
    }
    .info-box {
        background-color: #e3f2fd;
        padding: 1rem;
        border-radius: 10px;
        border-left: 5px solid #1E88E5;
    }
    .legend-box {
        background-color: #f5f5f5;
        padding: 1rem;
        border-radius: 10px;
        margin: 0.5rem 0;
    }
    .legend-item {
        display: flex;
        align-items: center;
        margin: 0.5rem 0;
    }
    .color-box {
        width: 30px;
        height: 30px;
        margin-right: 10px;
        border-radius: 5px;
        border: 2px solid #333;
    }
    </style>
''', unsafe_allow_html=True)

# Enhanced Color Classifier - Detects ALL shades of blue
class EnhancedColorClassifier:
    def __init__(self):
        # Multiple blue ranges in HSV to catch all shades
        self.blue_ranges = [
            # Light blue - sky blue, baby blue
            {
                'lower': np.array([90, 30, 100]),
                'upper': np.array([110, 255, 255]),
                'name': 'Light Blue'
            },
            # Medium blue - royal blue, cobalt
            {
                'lower': np.array([100, 50, 50]),
                'upper': np.array([130, 255, 255]),
                'name': 'Medium Blue'
            },
            # Dark blue - navy, midnight blue
            {
                'lower': np.array([100, 30, 20]),
                'upper': np.array([140, 255, 100]),
                'name': 'Dark Blue'
            },
            # Extended blue range for metallic car paints
            {
                'lower': np.array([95, 40, 40]),
                'upper': np.array([135, 255, 220]),
                'name': 'Metallic Blue'
            },
            # Cyan-blue - teal, turquoise
            {
                'lower': np.array([85, 40, 40]),
                'upper': np.array([95, 255, 255]),
                'name': 'Cyan Blue'
            }
        ]
        self.blue_threshold = 0.15

    def get_dominant_color(self, image):
        if image is None or image.size == 0:
            return None
        try:
            pixels = image.reshape((-1, 3))
            if len(pixels) > 10000:
                indices = np.random.choice(len(pixels), 10000, replace=False)
                pixels = pixels[indices]
            kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
            kmeans.fit(pixels)
            centers = kmeans.cluster_centers_.astype(np.uint8)
            labels = kmeans.labels_
            frequencies = np.bincount(labels)
            dominant_color = centers[np.argmax(frequencies)]
            return dominant_color
        except:
            return None

    def check_blue_percentage(self, image):
        if image is None or image.size == 0:
            return 0
        try:
            hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
            total_mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
            for blue_range in self.blue_ranges:
                mask = cv2.inRange(hsv, blue_range['lower'], blue_range['upper'])
                total_mask = cv2.bitwise_or(total_mask, mask)
            blue_pixels = np.sum(total_mask > 0)
            total_pixels = hsv.shape[0] * hsv.shape[1]
            percentage = blue_pixels / total_pixels
            return percentage
        except:
            return 0

    def is_blue(self, color):
        if color is None:
            return False
        try:
            color_hsv = cv2.cvtColor(np.uint8([[color]]), cv2.COLOR_BGR2HSV)[0][0]
            for blue_range in self.blue_ranges:
                lower = blue_range['lower']
                upper = blue_range['upper']
                if (lower[0] <= color_hsv[0] <= upper[0] and
                    lower[1] <= color_hsv[1] <= upper[1] and
                    lower[2] <= color_hsv[2] <= upper[2]):
                    return True
            return False
        except:
            return False

    def classify_color_advanced(self, car_roi):
        if car_roi is None or car_roi.size == 0:
            return "Other", 0

        blue_percentage = self.check_blue_percentage(car_roi)
        dominant_color = self.get_dominant_color(car_roi)
        is_dominant_blue = self.is_blue(dominant_color)
        has_blue_pixels = blue_percentage > self.blue_threshold

        if is_dominant_blue or has_blue_pixels or blue_percentage > 0.1:
            confidence = max(blue_percentage * 2, 0.5)
            if confidence > 0.5:
                return "Blue", confidence

        return "Other", 1 - min(blue_percentage * 2, 0.5)

# Traffic Analyzer
class TrafficAnalyzer:
    def __init__(self):
        with st.spinner("Loading AI Model... Please wait (this may take 1-2 minutes)."):
            try:
                self.model = YOLO('yolov8n.pt')
            except Exception as e:
                st.warning(f"Model load issue: {e}. Retrying...")
                self.model = YOLO('yolov8n.pt')
        self.color_classifier = EnhancedColorClassifier()
        st.success("Model loaded successfully")

    def detect_objects(self, image):
        results = self.model(image, conf=0.5, verbose=False)
        return results[0]

    def process_frame(self, frame):
        if frame is None:
            return None, 0, 0, 0, []

        processed_frame = frame.copy()
        results = self.detect_objects(processed_frame)

        blue_cars = 0
        other_cars = 0
        people_count = 0
        blue_confidences = []
        car_details = []

        if results.boxes is not None:
            for detection in results.boxes.data.tolist():
                x1, y1, x2, y2, conf, cls = detection
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                if int(cls) == 2:  # Car
                    car_roi = frame[y1:y2, x1:x2]
                    color, confidence = self.color_classifier.classify_color_advanced(car_roi)

                    # CRITICAL: RED rectangles for BLUE cars
                    if color == "Blue":
                        # RED rectangle for blue cars (BGR: 0,0,255)
                        cv2.rectangle(processed_frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
                        blue_cars += 1
                        blue_confidences.append(confidence)
                        label = f'BLUE CAR ({confidence*100:.0f}%)'
                        car_details.append({'color': 'Blue', 'confidence': confidence})
                        # Yellow text for visibility
                        cv2.putText(processed_frame, label, (x1, y1-25),
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
                    else:
                        # BLUE rectangle for other cars (BGR: 255,0,0)
                        cv2.rectangle(processed_frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
                        other_cars += 1
                        car_details.append({'color': 'Other', 'confidence': 0})
                        label = 'OTHER CAR'
                        cv2.putText(processed_frame, label, (x1, y1-25),
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

                    # Add color label on the box
                    cv2.putText(processed_frame, f'{color}', (x1, y1-10),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

                elif int(cls) == 0:  # Person
                    people_count += 1
                    # GREEN rectangle for people
                    cv2.rectangle(processed_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(processed_frame, 'PERSON', (x1, y1-10),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        # Add information overlay with colors
        info_text = [
            f'BLUE CARS: {blue_cars} (RED rectangles)',
            f'OTHER CARS: {other_cars} (BLUE rectangles)',
            f'TOTAL CARS: {blue_cars + other_cars}',
            f'PEOPLE: {people_count}'
        ]

        y_offset = 30
        for text in info_text:
            cv2.putText(processed_frame, text, (10, y_offset),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            y_offset += 30

        return processed_frame, blue_cars, other_cars, people_count, car_details

# Main app
def main():
    # Header
    st.markdown('<p class="main-header">Car Color Detection and Traffic Analysis System</p>', unsafe_allow_html=True)
    st.markdown('<p class="sub-header">Upload a traffic image to detect cars, identify blue vehicles, and count people</p>', unsafe_allow_html=True)

    # Color Legend
    st.markdown('''
    <div class="legend-box">
        <h4>Color Legend</h4>
        <div class="legend-item">
            <div class="color-box" style="background-color: #FF0000;"></div>
            <span><strong>RED rectangle</strong> = Blue car (All shades of blue)</span>
        </div>
        <div class="legend-item">
            <div class="color-box" style="background-color: #0000FF;"></div>
            <span><strong>BLUE rectangle</strong> = Other colored car</span>
        </div>
        <div class="legend-item">
            <div class="color-box" style="background-color: #00FF00;"></div>
            <span><strong>GREEN rectangle</strong> = Person</span>
        </div>
    </div>
    ''', unsafe_allow_html=True)

    st.markdown("---")

    # Sidebar
    with st.sidebar:
        st.markdown("### Upload Image")
        uploaded_file = st.file_uploader("Choose an image...", type=['jpg', 'jpeg', 'png', 'bmp', 'tiff'])

        st.markdown("---")
        st.markdown("### Detection Features")
        st.markdown('''
        - Detect all cars
        - Identify blue cars (RED boxes)
        - Identify other cars (BLUE boxes)
        - Count people (GREEN boxes)
        - All shades of blue detected
        ''')

        st.markdown("---")
        st.markdown("### Blue Shades Detected")
        st.markdown('''
        - Light Blue (Sky blue)
        - Medium Blue (Royal blue)
        - Dark Blue (Navy)
        - Metallic Blue
        - Cyan Blue (Teal)
        ''')

        st.markdown("---")
        st.markdown("### Statistics")
        st.markdown('''
        - Number of blue cars
        - Number of other cars
        - Total cars
        - People count
        ''')

    # Main content area
    if uploaded_file is not None:
        # Read the image
        file_bytes = np.asarray(bytearray(uploaded_file.read()), dtype=np.uint8)
        image = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Create columns for layout
        col1, col2 = st.columns([1, 1])

        with col1:
            st.markdown("### Original Image")
            st.image(image_rgb, use_container_width=True)

            # Image info
            st.markdown(f"**Filename:** {uploaded_file.name}")
            st.markdown(f"**Dimensions:** {image.shape[1]} x {image.shape[0]} pixels")

        # Process button
        if st.button("Analyze Traffic Scene", type="primary", use_container_width=True):
            with st.spinner("Analyzing image... Please wait."):
                # Initialize analyzer
                analyzer = TrafficAnalyzer()

                # Process image
                processed, blue_cars, other_cars, people, car_details = analyzer.process_frame(image)
                processed_rgb = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)

                # Display results in columns
                with col2:
                    st.markdown("### Analysis Results")
                    st.image(processed_rgb, use_container_width=True)

                    # Show color legend on results
                    st.markdown('''
                    <div style="font-size: 0.9rem; background-color: #f5f5f5; padding: 10px; border-radius: 5px;">
                        <span style="color: #FF0000;">■</span> RED = Blue car &nbsp;&nbsp;
                        <span style="color: #0000FF;">■</span> BLUE = Other car &nbsp;&nbsp;
                        <span style="color: #00FF00;">■</span> GREEN = Person
                    </div>
                    ''', unsafe_allow_html=True)

                # Statistics
                st.markdown("---")
                st.markdown("### Traffic Statistics")

                # Create metric cards
                metric_col1, metric_col2, metric_col3, metric_col4 = st.columns(4)

                with metric_col1:
                    st.markdown(f'''
                    <div class="stats-box">
                        <h2 style="color: #D32F2F; margin:0;">{blue_cars}</h2>
                        <p style="margin:0; color: #D32F2F;">Blue Cars</p>
                        <p style="margin:0; font-size: 0.8rem; color: #666;">(RED rectangles)</p>
                    </div>
                    ''', unsafe_allow_html=True)

                with metric_col2:
                    st.markdown(f'''
                    <div class="stats-box">
                        <h2 style="color: #1565C0; margin:0;">{other_cars}</h2>
                        <p style="margin:0; color: #1565C0;">Other Cars</p>
                        <p style="margin:0; font-size: 0.8rem; color: #666;">(BLUE rectangles)</p>
                    </div>
                    ''', unsafe_allow_html=True)

                with metric_col3:
                    st.markdown(f'''
                    <div class="stats-box">
                        <h2 style="color: #2E7D32; margin:0;">{people}</h2>
                        <p style="margin:0; color: #2E7D32;">People</p>
                        <p style="margin:0; font-size: 0.8rem; color: #666;">(GREEN rectangles)</p>
                    </div>
                    ''', unsafe_allow_html=True)

                with metric_col4:
                    total_cars = blue_cars + other_cars
                    st.markdown(f'''
                    <div class="stats-box">
                        <h2 style="color: #FF6F00; margin:0;">{total_cars}</h2>
                        <p style="margin:0; color: #FF6F00;">Total Cars</p>
                        <p style="margin:0; font-size: 0.8rem; color: #666;">Combined</p>
                    </div>
                    ''', unsafe_allow_html=True)

                # Detailed breakdown
                if blue_cars > 0:
                    st.markdown("---")
                    st.markdown("### Blue Car Details")
                    blue_count = 0
                    for i, detail in enumerate(car_details, 1):
                        if detail['color'] == 'Blue':
                            blue_count += 1
                            st.success(f"Car {blue_count}: Blue detected with {detail['confidence']*100:.1f}% confidence (RED rectangle)")

                # Download buttons
                st.markdown("---")
                st.markdown("### Download Results")

                col_download1, col_download2, col_download3 = st.columns(3)

                with col_download1:
                    # Convert processed image to bytes for download
                    processed_pil = Image.fromarray(processed_rgb)
                    import io
                    img_bytes = io.BytesIO()
                    processed_pil.save(img_bytes, format='PNG')
                    img_bytes = img_bytes.getvalue()

                    st.download_button(
                        label="Download Processed Image",
                        data=img_bytes,
                        file_name=f"processed_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png",
                        mime="image/png",
                        use_container_width=True
                    )

                with col_download2:
                    # Create summary text
                    summary = f'''
                    TRAFFIC ANALYSIS SUMMARY
                    {'='*50}
                    File: {uploaded_file.name}
                    Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

                    COLOR LEGEND:
                    - RED rectangles = Blue cars
                    - BLUE rectangles = Other cars
                    - GREEN rectangles = People

                    RESULTS:
                    {'-'*50}
                    Blue Cars  : {blue_cars}  (RED rectangles)
                    Other Cars : {other_cars}  (BLUE rectangles)
                    Total Cars : {blue_cars + other_cars}
                    People     : {people}  (GREEN rectangles)
                    {'='*50}
                    '''

                    st.download_button(
                        label="Download Summary",
                        data=summary,
                        file_name=f"summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt",
                        mime="text/plain",
                        use_container_width=True
                    )

                with col_download3:
                    # Original image download
                    original_pil = Image.fromarray(image_rgb)
                    orig_bytes = io.BytesIO()
                    original_pil.save(orig_bytes, format='PNG')
                    orig_bytes = orig_bytes.getvalue()

                    st.download_button(
                        label="Download Original",
                        data=orig_bytes,
                        file_name=f"original_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png",
                        mime="image/png",
                        use_container_width=True
                    )

    else:
        # Show placeholder when no image is uploaded
        st.markdown('''
            <div style="text-align: center; padding: 3rem 0; background-color: #f8f9fa; border-radius: 10px;">
                <h2>Upload an Image to Begin</h2>
                <p style="color: #666;">Click the <strong>"Browse files"</strong> button in the sidebar</p>
                <p style="color: #999; font-size: 0.9rem;">Supported formats: JPG, JPEG, PNG, BMP, TIFF</p>
                <br>
                <p style="color: #999; font-size: 0.9rem;">Best results with clear traffic scenes</p>
                <p style="color: #999; font-size: 0.9rem;">RED = Blue cars, BLUE = Other cars, GREEN = People</p>
            </div>
            ''', unsafe_allow_html=True)

if __name__ == "__main__":
    main()
""")

# Write the app to a file
with open('car_detection_app.py', 'w') as f:
    f.write(app_code)

print("Streamlit app created successfully")
print("Color Coding: RED rectangles for BLUE cars, BLUE rectangles for OTHER cars")

Streamlit app created successfully
Color Coding: RED rectangles for BLUE cars, BLUE rectangles for OTHER cars


In [27]:
# If running in Google Colab, this will create a public URL
!pip install pyngrok

from pyngrok import ngrok

# IMPORTANT: Authenticate ngrok with your authtoken.
# Get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
# Replace 'YOUR_AUTH_TOKEN' with your actual ngrok authtoken.
ngrok.set_auth_token('3J1wErsX1gKHKEbyoOQmgUcBHp5_41bdCXPe8CzXx4mCXUVVp')

# Kill any existing tunnels
!pkill -f streamlit

# Run streamlit in background
!nohup streamlit run car_detection_app.py --server.port 8501 &

# Create public URL
public_url = ngrok.connect(8501)
print(f"\n🚀 Your app is running at: {public_url}")

nohup: appending output to 'nohup.out'

🚀 Your app is running at: NgrokTunnel: "https://dormitory-thwarting-astrology.ngrok-free.dev" -> "http://localhost:8501"
